# TWYD End-to-End (Colab zero-install)
Train What You Deploy: Dense-LRC / CORE-LRC.
Repo: https://github.com/wenhui-ml/twyd (public, no token).
Core rule: NO pip install -e needed. twyd core = torch + transformers + safetensors only (Colab preinstalled). Just sys.path.insert(ROOT), then import twyd.


In [ ]:
import os
os.environ['TWYD_ROOT'] = os.environ.get('TWYD_ROOT', '') or '/content/drive/MyDrive/Papers/twyd'
ROOT = os.environ['TWYD_ROOT']
print('ROOT =', ROOT)
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())


## 0a. mount Drive (Colab only; local: skip)
Once per session. Outputs stay on Drive, survive restarts.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
except Exception as e:
    print('local run, skip mount:', e)


## 0b. fetch code if ROOT empty (public, no token)


In [ ]:
public repo clone, no token
import os, subprocess, shutil
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
REPO = 'https://github.com/wenhui-ml/twyd.git'
marker = os.path.join(ROOT, 'twyd', '__init__.py')
if os.path.isfile(marker):
    print('code present, skip clone')
else:
    os.makedirs(ROOT, exist_ok=True)
    tmp = ROOT + '.tmpclone'
    print('cloning', REPO)
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO, tmp], capture_output=True, text=True)
    print(((r.stdout or '') + (r.stderr or ''))[-400:])
    assert r.returncode == 0, 'clone failed'
    print('clone ok')


## 0c. zero-install wiring (THE FIX for your pip errors)


In [ ]:
import os, sys
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
for p in (ROOT, os.path.join(ROOT, 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('sys.path head:', sys.path[:2])
import twyd
import twyd.plugin as P
print('tuners:', sorted(P._TUNERS))
print('losses:', sorted(P._LOSSES))
print('zero-install import OK')


## 1. checks (pure Python)


In [ ]:
import os
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
need = ['tests/test_twyd_synthetic.py', 'scripts/cpu_smoke.py', 'twyd/tuner.py']
missing = [f for f in need if not os.path.isfile(os.path.join(ROOT, f))]
print('missing:', missing if missing else 'none')


In [ ]:
import subprocess, sys, os
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
r = subprocess.run([sys.executable, '-m', 'pytest', os.path.join(ROOT, 'tests'), '-q'], cwd=ROOT, capture_output=True, text=True)
print((r.stdout + r.stderr)[-800:])


### 1b. param gap plain rH vs dense/core rd_ff


In [ ]:
from twyd.config import TwydLRCConfig
from twyd.tuner import _build_factorized_student, _freeze_for_branch, _TwydWrapper
from tests.test_twyd_synthetic import _make_pair
for br in ['plain', 'dense', 'core']:
    t, s = _make_pair(24, 64, 12)
    cfg = TwydLRCConfig(branch=br, student_hidden=12)
    _build_factorized_student(s, t, cfg, branch=br)
    w = _TwydWrapper(s, t, cfg)
    _freeze_for_branch(w, cfg)
    n = sum(p.numel() for p in w.student.parameters() if p.requires_grad)
    print(br, n)


### 1c. CPU smoke (tiny-Llama real forward, ~2 min)


In [ ]:
import subprocess, sys, os
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
r = subprocess.run([sys.executable, os.path.join(ROOT, 'scripts', 'cpu_smoke.py'), '--steps', '10'], cwd=ROOT, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1200:])


## 2. train: project-direct python (no swift CLI needed for smoke)
Qwen0.5B smoke = scripts/train_dense_standalone.py (teacher + student built in-process).
Full swift pt path kept as reference text only.


In [ ]:
import os
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
print(open(os.path.join(ROOT, 'scripts', 'smoke_qwen05b.sh')).read())
print('direct run (GPU): python scripts/train_dense_standalone.py --teacher Qwen/Qwen2.5-0.5B --output outputs/dense-05b --steps 100')


### 2b. REAL TRAIN starts here (needs Colab T4 GPU)
Run gpu_check first. If cuda=False, stop and switch runtime. Qwen0.5B 100 steps ~10 min on T4, outputs to $ROOT/outputs/dense-05b.

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('STOP: switch Colab Runtime -> Change runtime type -> T4 GPU, then rerun from here')


### 2c. launch Qwen0.5B dense smoke (this cell IS training)
`STEPS=100` smoke, `STEPS=500` overnight. `TEACHER` env can point to local path to skip download.

In [ ]:
import os, subprocess, sys
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
OUT = os.path.join(ROOT, 'outputs', 'dense-05b')
TEACHER = os.environ.get('TEACHER', 'Qwen/Qwen2.5-0.5B')
STEPS = int(os.environ.get('STEPS', '100'))
DEVICE = 'cuda:0' if __import__('torch').cuda.is_available() else 'cpu'
print('teacher:', TEACHER, '| out:', OUT, '| steps:', STEPS, '| device:', DEVICE)
if DEVICE == 'cpu':
    print('REFUSE to train on CPU: Qwen0.5B needs GPU. Change runtime to GPU first.')
else:
    cmd = [sys.executable, os.path.join(ROOT, 'scripts', 'train_dense_standalone.py'), '--teacher', TEACHER, '--output', OUT, '--student-hidden', '448', '--seq-len', '256', '--batch', '1', '--grad-accum', '4', '--steps', str(STEPS), '--lr', '1e-4', '--temperature', '40.0', '--device', DEVICE]
    print('running:', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=ROOT)
    print('returncode:', r.returncode)


## 3. convert: merge to HF (direct import, no CLI)


In [ ]:
from twyd.merge import merged_state_dict
from tests.test_twyd_synthetic import _make_pair, _wrap
for br in ['dense', 'core', 'plain']:
    t, s = _make_pair(24, 64, 12)
    w, _ = _wrap(br, t, s, 12)
    sd = merged_state_dict(w)
    leaks = [k for k in sd if '.z_col' in k or '.z_perp' in k or 'basis_' in k]
    print(br, 'leak:', leaks, 'n:', len(sd))


## 4. eval: generation check via direct HF load (no swift eval install)
Fill CKPT with your merged dir.


In [ ]:
import os
ROOT = os.environ.get('TWYD_ROOT', '/content/drive/MyDrive/Papers/twyd')
CKPT = os.path.join(ROOT, 'outputs', 'dense-05b-deploy')
print('CKPT =', CKPT, 'exists:', os.path.isdir(CKPT))


## 5. fix table
| your error | cause | this notebook fix |
|---|---|---|
| _upstream/ms-swift not valid editable | gitignored, never pushed to GitHub | deleted that pip line; twyd core never imports swift (try/except fallback) |
| ROOT has no setup.py/pyproject | confusion: pyproject lives at repo root | no pip install at all; sys.path insert |
| No module named twyd | pip failed so import fails | 0c wires sys.path before any import |
| bash $ROOT not persisting | %%bash -s $ROOT needs shell var | all key steps now pure Python + os.environ |
